# ENN583 — Week 13 Practical, Part 1: Calibration under distribution shift

In this notebook you will evaluate a pretrained classifier on 350 ImageNet images and 350 matched ObjectNet images from the same seven classes. You will distinguish accuracy from confidence calibration, build reliability diagrams, fit temperature scaling without test leakage, and examine what happens when the input distribution changes. Data used to fit calibration will remain separate from data used to evaluate it.

## 0. Setup and data

In [ ]:
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import ResNet50_Weights, resnet50

# Fix random seeds so repeated runs use the same model-independent choices.
SEED = 583
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
# Support launching from the repository root or the Week 13 directory.
week13_candidates = [Path('.'), Path('practicals/Week_13')]
WEEK13_DIR = next(p for p in week13_candidates if (p / 'assets/imagenet_subset.zip').exists())
ARCHIVE_DIR = WEEK13_DIR / 'assets'
DATA_DIR = WEEK13_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)

# Extract each supplied archive once; later runs reuse the extracted folder.
for name in ('imagenet_subset', 'objectnet_subset'):
    destination = DATA_DIR / name
    if not destination.exists():
        print('Unpacking', name)
        with ZipFile(ARCHIVE_DIR / f'{name}.zip') as archive:
            archive.extractall(DATA_DIR)

weights = ResNet50_Weights.DEFAULT
# Resize every image to the input size expected by this practical, then
# apply the channel-wise normalisation used to train ResNet-50 on ImageNet.
# Direct resizing can change aspect ratio; keep that limitation in mind.
imagenet_means = (0.485, 0.456, 0.406)
imagenet_stds = (0.229, 0.224, 0.225)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224), antialias=True),
    transforms.Normalize(imagenet_means, imagenet_stds),
])
# Ignore notebook checkpoint copies accidentally stored inside the archives.
def is_course_image(path):
    path = Path(path)
    return '.ipynb_checkpoints' not in path.parts and path.suffix.lower() in {'.jpg', '.jpeg', '.png'}

imagenet_data = ImageFolder(DATA_DIR / 'imagenet_subset', transform=transform, is_valid_file=is_course_image)
objectnet_data = ImageFolder(DATA_DIR / 'objectnet_subset', transform=transform, is_valid_file=is_course_image)
assert imagenet_data.classes == objectnet_data.classes
assert len(imagenet_data) == len(objectnet_data) == 350
print(imagenet_data.classes)
print('ImageNet subset:', len(imagenet_data), 'ObjectNet subset:', len(objectnet_data))

## 1. A pretrained ImageNet classifier

The seven folders correspond to seven of ResNet-50's 1000 ImageNet outputs. We select those seven logits and apply a seven-way softmax. A **logit** is an unnormalised model score; softmax converts the selected scores into non-negative values that sum to one. This is a closed-set evaluation: it assumes every image belongs to one of the seven known classes and does not test whether the model can reject an unknown class.

In [ ]:
# Folder order: backpack, computer mouse, laptop computer, mobile phone,
# pen, wallet, water bottle. These are their ImageNet-1K output indices.
IMAGENET_INDICES = torch.tensor([414, 673, 620, 487, 418, 893, 898], device=DEVICE)
# eval() disables training-only behaviour such as BatchNorm updates.
model = resnet50(weights=weights).to(DEVICE).eval()

def collect_logits(dataset, batch_size=32):
    # Preserve dataset order so each output remains aligned with its label.
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    logits, targets = [], []
    # We evaluate a fixed pretrained model, so gradients would waste memory.
    with torch.no_grad():
        for images, labels in loader:
            # ResNet-50 returns one raw logit for each of 1000 ImageNet classes.
            full_logits = model(images.to(DEVICE))
            # Keep only the seven classes represented in this practical.
            logits.append(full_logits.index_select(1, IMAGENET_INDICES).cpu())
            targets.append(labels)
    return torch.cat(logits), torch.cat(targets)

imagenet_logits, imagenet_targets = collect_logits(imagenet_data)
objectnet_logits, objectnet_targets = collect_logits(objectnet_data)
for name, current_logits, targets in [
    ('ImageNet', imagenet_logits, imagenet_targets),
    ('ObjectNet', objectnet_logits, objectnet_targets),
]:
    accuracy = current_logits.argmax(1).eq(targets).float().mean().item()
    print(f'{name:9s} accuracy: {accuracy:.3f}')

## 2. Reliability diagrams and ECE

For each image, confidence is the largest of the seven softmax values. We group predictions into ten confidence intervals. Each upper-panel bar reports the fraction correct within one interval; the lower panel shows how many predictions support that estimate. Empty bins contain no evidence and are left blank. **Expected calibration error (ECE)** is the count-weighted average of the absolute gap between mean confidence and accuracy in each populated bin. ECE depends on the sample size and bin boundaries, so interpret it together with the diagram and count histogram.

In [ ]:
def calibration_bins(logits, targets, n_bins=10):
    # Convert seven raw class scores into values that sum to one per image.
    probabilities = logits.softmax(1)
    confidence, prediction = probabilities.max(1)
    # correct is 1.0 for a correct prediction and 0.0 for an error.
    correct = prediction.eq(targets).float()
    # Eleven edges define ten equal-width intervals over confidence [0, 1].
    edges = torch.linspace(0, 1, n_bins + 1)
    accuracy = np.full(n_bins, np.nan)
    mean_confidence = np.full(n_bins, np.nan)
    counts = np.zeros(n_bins, dtype=int)
    ece = 0.0
    for i, (lower, upper) in enumerate(zip(edges[:-1], edges[1:])):
        # Use half-open intervals, except the final bin includes confidence 1.0.
        mask = (confidence >= lower) & ((confidence < upper) if i < n_bins - 1 else (confidence <= upper))
        counts[i] = int(mask.sum())
        if counts[i]:
            accuracy[i] = correct[mask].mean().item()
            mean_confidence[i] = confidence[mask].mean().item()
            # Large bins contribute more to ECE than bins containing few images.
            ece += counts[i] / len(targets) * abs(accuracy[i] - mean_confidence[i])
    return edges.numpy(), accuracy, mean_confidence, counts, ece

def plot_calibration(logits, targets, title, axes=None, n_bins=10):
    edges, accuracy, mean_confidence, counts, ece = calibration_bins(logits, targets, n_bins)
    if axes is None:
        _, axes = plt.subplots(2, 1, figsize=(5, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
    # Plot bars at fixed bin centres; NaN leaves unsupported bins empty.
    centres = (edges[:-1] + edges[1:]) / 2
    width = 0.85 / n_bins
    axes[0].bar(centres, accuracy, width=width, edgecolor='black', alpha=.7)
    axes[0].plot([0, 1], [0, 1], 'k--', label='perfect calibration')
    axes[0].set(ylabel='Observed accuracy', title=f'{title} (ECE={ece:.3f})', ylim=(0, 1))
    axes[0].grid(axis='y', alpha=.3); axes[0].legend()
    axes[1].bar(centres, counts, width=width, edgecolor='black', color='tab:gray')
    axes[1].set(xlabel='Confidence', ylabel='Count', xlim=(0, 1))
    return ece

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex='col', gridspec_kw={'height_ratios': [3, 1]})
imagenet_ece = plot_calibration(imagenet_logits, imagenet_targets, 'ImageNet subset', axes[:, 0])
objectnet_ece = plot_calibration(objectnet_logits, objectnet_targets, 'ObjectNet subset', axes[:, 1])
plt.tight_layout(); plt.show()

**Discuss:** How do accuracy, confidence distribution, and calibration change on ObjectNet? Which bins are supported by enough examples to interpret? Why does an OOD or shifted example not necessarily receive low confidence?

## 3. Can temperature scaling improve the model further?

The reliability diagrams above suggest that ResNet-50 is already reasonably well calibrated on this ImageNet subset: its confidence and observed accuracy are fairly closely aligned. Even so, let us see whether **temperature scaling** can improve its confidence estimates further. Temperature scaling learns one positive number, $T$, and divides every class logit by it before softmax. It changes how sharp the probabilities are without changing which class has the largest logit.

To avoid evaluating the method on the same examples used to fit it, we split each ImageNet class deterministically: 25 images per class fit the temperature and the other 25 evaluate it. ObjectNet remains a separate shifted evaluation set. Keep your expectations modest: this is a small subset, the original model is already quite well calibrated, and the fitted scalar minimises negative log-likelihood (NLL) rather than the binned ECE shown in the diagram. Any improvement may therefore be small and sensitive to the particular split. Transfer to ObjectNet is even less certain because distribution shift can change the relationship between confidence and correctness.

In [ ]:
# Build a class-balanced split. Even-positioned examples fit the temperature;
# odd-positioned examples are untouched until final evaluation.
calibration_mask = torch.zeros(len(imagenet_targets), dtype=torch.bool)
for class_index in range(len(imagenet_data.classes)):
    class_positions = torch.where(imagenet_targets == class_index)[0]
    calibration_mask[class_positions[::2]] = True
evaluation_mask = ~calibration_mask
print('Calibration:', int(calibration_mask.sum()), 'Evaluation:', int(evaluation_mask.sum()))

# Optimise log(T), then exponentiate it, so temperature T is always positive.
log_temperature = nn.Parameter(torch.zeros(()))
optimiser = torch.optim.LBFGS([log_temperature], lr=.1, max_iter=100)
loss_fn = nn.CrossEntropyLoss()

def objective():
    optimiser.zero_grad()
    # T > 1 softens probabilities; T < 1 sharpens them. Dividing every
    # class logit by the same positive T cannot change the winning class.
    loss = loss_fn(imagenet_logits[calibration_mask] / log_temperature.exp(), imagenet_targets[calibration_mask])
    loss.backward()
    return loss

# LBFGS may call objective several times while searching for the best scalar.
optimiser.step(objective)
temperature = log_temperature.detach().exp().item()
print(f'Temperature: {temperature:.3f}')
assert torch.equal(imagenet_logits.argmax(1), (imagenet_logits / temperature).argmax(1))

In [ ]:
# Evaluate only after fitting is complete; these examples did not set T.
heldout_logits = imagenet_logits[evaluation_mask]
heldout_targets = imagenet_targets[evaluation_mask]
_, _, _, _, imagenet_before = calibration_bins(heldout_logits, heldout_targets)
_, _, _, _, imagenet_after = calibration_bins(heldout_logits / temperature, heldout_targets)
_, _, _, _, objectnet_before = calibration_bins(objectnet_logits, objectnet_targets)
_, _, _, _, objectnet_after = calibration_bins(objectnet_logits / temperature, objectnet_targets)
print(f'Held-out ImageNet ECE: {imagenet_before:.3f} -> {imagenet_after:.3f}')
print(f'ObjectNet ECE:          {objectnet_before:.3f} -> {objectnet_after:.3f}')
print(f'Held-out ImageNet NLL: {loss_fn(heldout_logits, heldout_targets):.3f} -> ' 
      f'{loss_fn(heldout_logits / temperature, heldout_targets):.3f}')
print(f'ObjectNet NLL:          {loss_fn(objectnet_logits, objectnet_targets):.3f} -> ' 
      f'{loss_fn(objectnet_logits / temperature, objectnet_targets):.3f}')

**Interpret the result:** Do not assume the numbers must improve. If held-out ImageNet ECE or NLL stays similar—or becomes slightly worse—the calibration subset was too small and the original model left little systematic miscalibration to correct. If ObjectNet becomes worse, the ImageNet-fitted temperature has encountered a different confidence–error relationship. Temperature scaling only rescales existing scores: it cannot detect distribution shift, change a predicted class, recover missing visual evidence, or make an incorrect prediction correct.

## 4. Seeing temperature scaling in action: controlled overconfidence

As we saw above, applying temperature scaling to an already well-calibrated model on a small dataset produces only a subtle result, and the fitted temperature may not transfer to shifted data. That is realistic, but it makes the basic mechanism difficult to see clearly. We will therefore create a controlled example in which the classifier is **deliberately badly calibrated**, then check whether temperature scaling can repair the known problem.

We make the classifier overconfident by multiplying every ObjectNet logit by 3. Because multiplication by a positive constant does not change the ordering of the logits, every predicted class—and therefore the accuracy—stays exactly the same. Softmax probabilities nevertheless become much sharper. We fit a new temperature on half of each ObjectNet class and evaluate it on the untouched half. This gives us a clear before/after demonstration of calibration working in action, which the small, already well-calibrated ImageNet subset could not show as clearly. The expected held-out change is approximately **ECE 0.205 to 0.086** and **NLL 1.560 to 0.920**.

In [ ]:
# Create a known failure mode for a controlled experiment. Multiplying all
# logits by a positive constant preserves argmax predictions but makes the
# softmax distribution sharper and usually more confident.
LOGIT_SHARPNESS = 3.0
overconfident_logits = objectnet_logits * LOGIT_SHARPNESS
assert torch.equal(objectnet_logits.argmax(1), overconfident_logits.argmax(1))

# Use disjoint, class-balanced halves for fitting and evaluation.
object_calibration_mask = torch.zeros(len(objectnet_targets), dtype=torch.bool)
for class_index in range(len(objectnet_data.classes)):
    class_positions = torch.where(objectnet_targets == class_index)[0]
    object_calibration_mask[class_positions[::2]] = True
object_evaluation_mask = ~object_calibration_mask

# Fit an independent temperature for this controlled experiment.
controlled_log_temperature = nn.Parameter(torch.zeros(()))
controlled_optimiser = torch.optim.LBFGS([controlled_log_temperature], lr=.1, max_iter=100)

def controlled_objective():
    controlled_optimiser.zero_grad()
    loss = loss_fn(
        overconfident_logits[object_calibration_mask] / controlled_log_temperature.exp(),
        objectnet_targets[object_calibration_mask],
    )
    loss.backward()
    return loss

# Fit only one scalar. The classifier weights and predicted classes stay fixed.
controlled_optimiser.step(controlled_objective)
controlled_temperature = controlled_log_temperature.detach().exp().item()
print(f'Injected logit multiplier: {LOGIT_SHARPNESS:.1f}')
print(f'Learned temperature:       {controlled_temperature:.3f}')

In [ ]:
# The untouched half measures whether calibration generalises beyond its fit set.
controlled_test_logits = overconfident_logits[object_evaluation_mask]
controlled_test_targets = objectnet_targets[object_evaluation_mask]
controlled_scaled_logits = controlled_test_logits / controlled_temperature

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex='col', gridspec_kw={'height_ratios': [3, 1]})
plot_calibration(controlled_test_logits, controlled_test_targets, 'Deliberately overconfident', axes[:, 0])
plot_calibration(controlled_scaled_logits, controlled_test_targets, 'After temperature scaling', axes[:, 1])
plt.tight_layout(); plt.show()

_, _, _, _, controlled_ece_before = calibration_bins(controlled_test_logits, controlled_test_targets)
_, _, _, _, controlled_ece_after = calibration_bins(controlled_scaled_logits, controlled_test_targets)
print(f'Held-out controlled ECE: {controlled_ece_before:.3f} -> {controlled_ece_after:.3f}')
print(f'Held-out controlled NLL: {loss_fn(controlled_test_logits, controlled_test_targets):.3f} -> ' 
      f'{loss_fn(controlled_scaled_logits, controlled_test_targets):.3f}')

**Interpret the result:** Multiplying all logits by 3 leaves their ordering—and therefore accuracy—unchanged, but exaggerates differences passed to softmax. A fitted temperature near 3 approximately reverses that artificial sharpening, so held-out ECE and NLL should improve visibly. This controlled result verifies the mechanism. It is not evidence that arbitrary deployment shift can be calibrated away.

## 5. Selective prediction and risk–coverage

The previous sections asked whether confidence values match observed accuracy on average. Even a well-calibrated model will still make mistakes, however, and calibration alone does not tell a deployed system when it should answer and when it should defer. In a safety-relevant application, we may prefer the model to **abstain** on uncertain inputs and pass them to a fallback such as another sensor, another camera view, a slower operating mode, or a human operator. This is called **selective prediction**.

There is a trade-off. If the model answers every input, **coverage** is 100%, but it includes both easy and difficult cases. Raising the confidence threshold rejects more cases, reducing coverage. If confidence ranks predictions usefully, the remaining answered cases should be safer: their **selective risk**, defined here as the error rate among answered predictions, should fall. A risk–coverage curve measures this trade-off over every possible confidence threshold.

We will sort predictions from most to least confident and progressively include less-confident cases. The left side of the curve contains only the most confident predictions; the right endpoint includes the entire dataset and therefore equals the ordinary error rate. We compare held-out ImageNet with ObjectNet to ask whether confidence remains a useful ranking signal after distribution shift. A curve that looks good on ImageNet may degrade on ObjectNet, so an abstention policy selected on one distribution is not automatically valid on another.

In [ ]:
def risk_coverage(logits, targets):
    # Confidence is used only to rank predictions from most to least trusted.
    probability = logits.softmax(1)
    confidence, prediction = probability.max(1)
    order = confidence.argsort(descending=True)
    # Accumulate errors as progressively less-confident predictions are accepted.
    errors = prediction[order].ne(targets[order]).float()
    # Prefix length gives the number answered at each possible coverage.
    answered = torch.arange(1, len(errors) + 1)
    return (answered / len(errors)).numpy(), (errors.cumsum(0) / answered).numpy()

fig, axis = plt.subplots(figsize=(6, 4))
for name, current_logits, targets in [
    ('held-out ImageNet', imagenet_logits[evaluation_mask], imagenet_targets[evaluation_mask]),
    ('ObjectNet shift', objectnet_logits, objectnet_targets),
]:
    # Use the same ImageNet-fitted temperature for both distributions.
    coverage, risk = risk_coverage(current_logits / temperature, targets)
    axis.plot(coverage, risk, label=name)
axis.set(xlabel='Coverage (fraction answered)', ylabel='Selective risk (error among answered)', xlim=(0, 1), ylim=(0, None))
axis.grid(alpha=.3); axis.legend(); plt.show()

**Interpret the result:** First compare the right endpoints; these are the overall error rates when the model answers everything. Then move left and ask whether risk falls as coverage decreases. A useful confidence ranking produces low risk among the most confident predictions. If ObjectNet remains above ImageNet at the same coverage, distribution shift has made abstention less effective.

**Discuss:** What coverage would you choose if an incorrect answer were much more costly than an abstention? Would you use the same operating point for ImageNet and ObjectNet? What additional validation data and information about real-world error costs would you need before choosing a deployment threshold?

## 6. Take-away

A reliability diagram needs enough examples in each confidence region. ECE compresses the diagram into one bin-dependent number and should always be reported with the binning rule and sample count. Temperature scaling changes confidence, not correctness. Risk–coverage evaluates whether confidence usefully ranks predictions for abstention, but an operating point chosen on one distribution need not transfer under shift.

Continue to the warehouse notebook to test these ideas in a robotics-specific setting and design a safe fallback.